Tools

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

model=init_chat_model("groq:qwen/qwen3.6-27b")
response=model.invoke("why do parrots talks ?")
response

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Understand User Question**: The user asks "why do parrots talks ?" (Note: grammatical error "talks" should be "talk", but the meaning is clear). They want to know the reason behind parrots\' ability to mimic human speech.\n\n2.  **Identify Key Concepts**:\n   - Parrot vocalization/mimicry\n   - Biological/anatomical basis (syrinx, brain structure)\n   - Evolutionary/social reasons (communication, bonding, survival)\n   - Cognitive aspects (intelligence, learning)\n   - Domestication vs. wild behavior\n\n3.  **Research/Verify Facts** (Internal Knowledge):\n   - Parrots have a specialized vocal organ called the syrinx, which allows them to produce a wide range of sounds.\n   - They have a specialized brain region (song system) that\'s highly developed for vocal learning.\n   - In the wild, parrots don\'t typically "talk" to humans; they mimic sounds from their flock for social bonding, communication, and survival.\n   - M

In [ ]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """get the weather"""
    return f"It's sunny in this {location}"

model_with_tools=model.bind_tools([get_weather])


_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.17'}}, client=<groq.resources.chat.completions.Completions object at 0x00000164D9277B60>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000164D94B06E0>, model_name='qwen/qwen3.6-27b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'get_weather', 'description': 'get the weather', 'parameters': {'properties': {'location': {'type': 'string'}}, 'required': ['location'], 'type': 'object'}}}]}, config={}, config_factories=[])

In [7]:
response=model_with_tools.invoke("what's the weather in boston?")
print(response)
for tool_call in response.tool_calls:
    print(f"tool : {tool_call['name']}")
    print(f"tool : {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'Thinking Process:\n1.  Identify the user\'s intent: The user is asking for the weather.\n2.  Identify the location: Boston.\n3.  Check available tools: `get_weather` function is available.\n4.  Call the `get_weather` function with the location "Boston".\n5.  Wait for the tool response.\n6.  Format the response for the user.\n\nLet\'s call the tool.✅\n', 'tool_calls': [{'id': 'dsnd03p1h', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 122, 'prompt_tokens': 273, 'total_tokens': 395, 'completion_time': 0.267498874, 'completion_tokens_details': {'reasoning_tokens': 94}, 'prompt_time': 0.021241906, 'prompt_tokens_details': None, 'queue_time': 0.044314878, 'total_time': 0.28874078}, 'model_name': 'qwen/qwen3.6-27b', 'system_fingerprint': 'fp_fff3b79855', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'mode

tool execution loop

In [ ]:
#step 1 Model generates tool calls
messages=[{"role":"user","content":"what is the weather in New York ? "}]
ai_msg=model_with_tools.invoke(messages)
messages.append(ai_msg)

#Step 2 Execute tools and collect result
for tool_call in ai_msg.tool_calls:
    tool_result=get_weather.invoke(tool_call)
    messages.append(tool_result)

#step3 Pass result back to model for final response
final_resp=model_with_tools.invoke(messages)
print(final_resp.text)    
messages
final_resp

It is currently sunny in New York.


AIMessage(content='It is currently sunny in New York.', additional_kwargs={'reasoning_content': 'The user asked for the weather in New York.\nI called the `get_weather` function with the location "New York".\nThe function returned "It\'s sunny in this New York".\nI should answer the user\'s question using this information.\n'}, response_metadata={'token_usage': {'completion_tokens': 63, 'prompt_tokens': 324, 'total_tokens': 387, 'completion_time': 0.120092868, 'completion_tokens_details': {'reasoning_tokens': 52}, 'prompt_time': 0.022752147, 'prompt_tokens_details': None, 'queue_time': 0.045025543, 'total_time': 0.142845015}, 'model_name': 'qwen/qwen3.6-27b', 'system_fingerprint': 'fp_49d6b1859d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a05e46-9f62-7f62-a229-9bc3e789495d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 324, 'output_tokens': 63, 'total_tokens': 387, 'output_token_details': 